# Image Classification using Convolutional Neural Networks (CNN) on CIFAR-10

### What was learned & implemented in this notebook:
1. **CNN Design Principles**:
   - Staged spatial feature extraction using convolutional layers (`nn.Conv2d`) with `kernel_size=3` and padding.
   - Downsampled activation maps using Max Pooling (`nn.MaxPool2d`) with a stride of 2.
2. **Data Pipeline Optimization**:
   - Bypassed slow, unstable downloads from the Toronto server by integrating Hugging Face's `datasets` API.
   - Configured custom PyTorch `Dataset` wraps to preprocess PIL images into normalized PyTorch tensors.
3. **Dense Classification Architecture**:
   - Flattened features (channel size × spatial dimensions) and mapped them using sequential linear layers (`nn.Linear`) to output class logits.
4. **Evaluation Loop & Debugging**:
   - Fixed a crucial accumulator bug in `total_labels` calculation by using `+=` addition instead of standard assignment (`=`), restoring correct validation accuracy metrics.

In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim 
import torchvision
from torchvision.datasets import CIFAR10 

In [4]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset = CIFAR10(root ="./data",train = True, download = True, transform = transform)
testset = CIFAR10(root ="./data",train = False, download = True, transform = transform)

  0%|          | 98.3k/170M [00:30<14:30:14, 3.26kB/s]


KeyboardInterrupt: 

In [5]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# 1. Load the CIFAR-10 dataset from Hugging Face (downloads in seconds!)
hf_dataset = load_dataset("uoft-cs/cifar10")

# Define the transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Create a custom PyTorch dataset wrapper for Hugging Face images
class HuggingFaceCIFAR10(Dataset):
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['img']  # This is a PIL Image
        label = item['label']
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# 3. Instantiate train and test sets
trainset = HuggingFaceCIFAR10(hf_dataset['train'], transform=transform)
testset = HuggingFaceCIFAR10(hf_dataset['test'], transform=transform)

# 4. Create your DataLoader loaders exactly as before
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)


**BUILD CNN**

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x= self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layers(x)
        return x 
        

In [7]:
model = CNN()

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### TRAIN CNN

In [11]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss= 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch= {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")
        

epoch= 1/10 & loss = 0.8280044017774065
epoch= 2/10 & loss = 0.6683546488394823
epoch= 3/10 & loss = 0.5579380053464714
epoch= 4/10 & loss = 0.4533350336201051
epoch= 5/10 & loss = 0.3652670359062722
epoch= 6/10 & loss = 0.27944006890896944
epoch= 7/10 & loss = 0.21637358635549656
epoch= 8/10 & loss = 0.165021220517471
epoch= 9/10 & loss = 0.13998190827115112
epoch= 10/10 & loss = 0.10852977648124937


In [17]:
correct_labels = 0
total_labels =  0
model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy = {correct_labels/total_labels *100}")

accuracy = 75.36
